# WOS Fine-Tuning — Gemma 2-27B (Colab)

This notebook trains the **WOS Gemma 2-27B** coding and meeting models.

**Requirements:**
- Colab Pro or Pro+ (need A100 40GB GPU)
- HuggingFace account with a Read+Write token
- ~2-3 hours per model

**Models this notebook trains:**
- `thejesraj/wos-coding-gemma` — coding assistant
- `thejesraj/wos-meeting-gemma` — meeting intelligence

---
**Step 1:** Run the GPU check cell below first. Make sure you see **A100**.

In [ ]:
# STEP 1: Verify GPU — must be A100
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout)
if 'A100' not in result.stdout:
    print('WARNING: Not an A100! Go to Runtime → Change runtime type → GPU → A100')
    print('You need Colab Pro for A100 access.')
else:
    print('GPU OK!')

In [ ]:
# STEP 2: Install packages (~5 min)
# Run this cell and wait for it to finish before continuing
!pip install -q torch==2.4.0 torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.46.3 trl==0.12.2 peft bitsandbytes accelerate datasets huggingface_hub boto3
print('Packages installed!')

In [ ]:
# STEP 3: Clone the WOS repo
import os
%cd /content
!git clone https://github.com/Gthejesraj/WOS.git
%cd /content/WOS/training
print('Repo cloned!')

In [ ]:
# STEP 4: Set your HuggingFace token
# Option A — Colab Secrets (recommended, keeps token hidden):
#   Click the key icon on the left sidebar → Add secret → Name: HF_TOKEN → Value: your_token
#   Then run this cell:
from google.colab import userdata
import os
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets!')
except Exception:
    # Option B — paste token directly (less safe, don't share this notebook after)
    os.environ['HF_TOKEN'] = 'paste_your_hf_token_here'
    print('HF_TOKEN set manually.')

from huggingface_hub import login, whoami
login(token=os.environ['HF_TOKEN'])
print(f"Logged in as: {whoami()['name']}")

---
## Train CODING model
Run the 3 cells below **in order**. Takes ~2-3 hours total.

In [ ]:
# CODING Step 1: Download dataset (~10 min)
%cd /content/WOS/training
!python datasets/coding/download.py
!python datasets/toolcalling/download.py
print('Datasets ready!')

In [ ]:
# CODING Step 2: Train (~2-3 hrs)
# This cell will run for hours — keep the Colab tab open
%cd /content/WOS/training
!python finetune/train.py --model coding --base gemma --with-tools

In [ ]:
# CODING Step 3: Verify model on HuggingFace
from huggingface_hub import model_info
try:
    info = model_info('thejesraj/wos-coding-gemma')
    print(f'Model live! thejesraj/wos-coding-gemma ({info.safetensors.total} shards)')
except Exception as e:
    print(f'Not found yet: {e}')

---
## Train MEETING model
Only run this after the Coding model finishes. Same 3 steps.

In [ ]:
# MEETING Step 1: Download dataset (~5 min)
%cd /content/WOS/training
!python datasets/meeting/download.py
# toolcalling already downloaded above, skip if already done
import os
if not os.path.exists('datasets/toolcalling/processed/train_split.jsonl'):
    !python datasets/toolcalling/download.py
print('Datasets ready!')

In [ ]:
# MEETING Step 2: Train (~1-2 hrs)
%cd /content/WOS/training
!python finetune/train.py --model meeting --base gemma --with-tools

In [ ]:
# MEETING Step 3: Verify model on HuggingFace
from huggingface_hub import model_info
try:
    info = model_info('thejesraj/wos-meeting-gemma')
    print(f'Model live! thejesraj/wos-meeting-gemma')
except Exception as e:
    print(f'Not found yet: {e}')

---
## If something goes wrong

- **`CUDA out of memory`** → Runtime → Restart runtime → run from Step 2 again
- **`ModuleNotFoundError`** → re-run the install cell (Step 2)
- **Session disconnected mid-training** → Re-run from the training cell — it'll restart from the latest checkpoint automatically
- **`Repository not found` on HF push** → Make sure your token has Write permission

Send the final output screenshot to Thejes when done.